# Sales & Business Performance Analytics — EDA
**Role:** Member 3 — Exploratory Data Analysis  
**Dataset:** `data/raw/superstore_dataset2011-2015.csv`

## Objective
Inspect data quality, validate dates, calculate baseline KPIs, and explore sales and profit patterns. This notebook does not overwrite the raw CSV. Original date strings are retained; parsed dates are stored in separate columns.

## 1. Imports and locate the dataset

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "superstore_dataset2011-2015.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Open this notebook from the project root or notebooks folder, "
        "or update DATA_PATH to your CSV location."
    )

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)

## 2. Load raw data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
display(df.head())

## 3. Dataset structure

In [ ]:
print("Columns:")
for column in df.columns:
    print("-", column)
print("\nData types and non-null counts:")
df.info()
display(df.describe(include="all").T)

## 4. Data quality

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
})
missing_summary = missing_summary[missing_summary["missing_count"] > 0].sort_values(
    "missing_count", ascending=False
)

print("Total exact duplicate rows:", df.duplicated().sum())
print("\nColumns with missing values:")
display(missing_summary)
for col in ["Order ID", "Customer ID", "Product ID"]:
    if col in df.columns:
        print(f"Unique {col}: {df[col].nunique():,}")
print("\nRepeated order/customer IDs can be normal in transaction data.")
print("Do not automatically remove missing postal codes or negative profits.")

## 5. Parse and validate dates

The investigation found slash-separated and hyphen-separated dates. Use `dayfirst=True` with mixed-format parsing, while retaining the original strings. A successful check supports this interpretation but does not prove every source date is historically correct.

In [ ]:
required_dates = ["Order Date", "Ship Date"]
missing_columns = [c for c in required_dates if c not in df.columns]
if missing_columns:
    raise KeyError(f"Missing required date columns: {missing_columns}")

row_count_before = len(df)
df["Order Date Parsed"] = pd.to_datetime(
    df["Order Date"], format="mixed", dayfirst=True, errors="coerce"
)
df["Ship Date Parsed"] = pd.to_datetime(
    df["Ship Date"], format="mixed", dayfirst=True, errors="coerce"
)

order_unparsed = int(df["Order Date Parsed"].isna().sum())
ship_unparsed = int(df["Ship Date Parsed"].isna().sum())
shipping_before_order = int(
    (df["Ship Date Parsed"] < df["Order Date Parsed"]).sum()
)

print("Rows:", len(df))
print("Unparsed order dates:", order_unparsed)
print("Unparsed ship dates:", ship_unparsed)
print("Shipping dates before order dates:", shipping_before_order)
print("Order date range:", df["Order Date Parsed"].min(), "to", df["Order Date Parsed"].max())
print("Ship date range:", df["Ship Date Parsed"].min(), "to", df["Ship Date Parsed"].max())

assert len(df) == row_count_before, "Unexpected row-count change."
assert order_unparsed == 0, "Investigate unparsed order dates before continuing."
assert ship_unparsed == 0, "Investigate unparsed ship dates before continuing."
assert shipping_before_order == 0, "Investigate date interpretation before continuing."
print("Date validation passed.")

## 6. Numeric summaries and distributions

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
display(df[numeric_cols].describe().T)

for col in [c for c in ["Sales", "Profit", "Quantity", "Discount", "Shipping Cost"] if c in df.columns]:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    sns.histplot(df[col].dropna(), bins=40, ax=axes[0])
    axes[0].set_title(f"{col} distribution")
    sns.boxplot(x=df[col], ax=axes[1])
    axes[1].set_title(f"{col} boxplot")
    plt.tight_layout()
    plt.show()

print("Treat unusual values as items to investigate, not automatic errors.")

## 7. Baseline business KPIs

In [ ]:
required = ["Sales", "Profit", "Quantity", "Order ID", "Customer ID"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"Missing KPI columns: {missing}")

total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
unique_orders = df["Order ID"].nunique()
unique_customers = df["Customer ID"].nunique()
aov = total_sales / unique_orders if unique_orders else np.nan
margin = total_profit / total_sales * 100 if total_sales else np.nan

kpi_summary = pd.DataFrame({
    "Metric": ["Total Sales", "Total Profit", "Total Quantity", "Unique Orders",
               "Unique Customers", "Average Order Value", "Profit Margin (%)"],
    "Value": [total_sales, total_profit, df["Quantity"].sum(), unique_orders,
              unique_customers, aov, margin]
})
display(kpi_summary)
print("Confirm currency conventions before external reporting.")

## 8. Category and sub-category performance

In [ ]:
if {"Category", "Sales", "Profit", "Quantity"}.issubset(df.columns):
    category_perf = df.groupby("Category", dropna=False).agg(
        Sales=("Sales", "sum"), Profit=("Profit", "sum"),
        Quantity=("Quantity", "sum"), Records=("Category", "size")
    ).sort_values("Sales", ascending=False)
    display(category_perf)
    category_perf[["Sales", "Profit"]].plot(kind="bar", figsize=(9, 4))
    plt.title("Sales and Profit by Category")
    plt.ylabel("Amount")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

if {"Sub-Category", "Sales", "Profit"}.issubset(df.columns):
    subcategory_perf = df.groupby("Sub-Category", dropna=False).agg(
        Sales=("Sales", "sum"), Profit=("Profit", "sum")
    ).sort_values("Sales", ascending=False)
    display(subcategory_perf)
    subcategory_perf[["Sales", "Profit"]].sort_values("Sales").plot(
        kind="barh", figsize=(10, 7)
    )
    plt.title("Sales and Profit by Sub-Category")
    plt.xlabel("Amount")
    plt.tight_layout()
    plt.show()

## 9. Market, region, country, segment, shipping, and priority

In [ ]:
for col in ["Market", "Region", "Country", "Segment", "Ship Mode", "Order Priority"]:
    if col in df.columns:
        summary = df.groupby(col, dropna=False).agg(
            Sales=("Sales", "sum"), Profit=("Profit", "sum"), Records=(col, "size")
        ).sort_values("Sales", ascending=False)
        print(f"\nPerformance by {col}:")
        display(summary.head(20))
        summary[["Sales", "Profit"]].head(15).plot(kind="bar", figsize=(10, 4))
        plt.title(f"Sales and Profit by {col} (top 15 by sales)")
        plt.ylabel("Amount")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

## 10. Customer and product analysis

In [ ]:
if {"Customer ID", "Sales", "Profit", "Order ID"}.issubset(df.columns):
    customer_perf = df.groupby("Customer ID", dropna=False).agg(
        Sales=("Sales", "sum"), Profit=("Profit", "sum"),
        Orders=("Order ID", "nunique")
    ).sort_values("Sales", ascending=False)
    print("Top 10 customers by sales:")
    display(customer_perf.head(10))

if {"Product Name", "Sales", "Profit", "Quantity"}.issubset(df.columns):
    product_perf = df.groupby("Product Name", dropna=False).agg(
        Sales=("Sales", "sum"), Profit=("Profit", "sum"),
        Quantity=("Quantity", "sum")
    ).sort_values("Sales", ascending=False)
    print("Top 10 products by sales:")
    display(product_perf.head(10))

## 11. Monthly and yearly trends

In [ ]:
trend_df = df.dropna(subset=["Order Date Parsed"]).copy()
trend_df["Order Month"] = trend_df["Order Date Parsed"].dt.to_period("M").dt.to_timestamp()
trend_df["Order Year"] = trend_df["Order Date Parsed"].dt.year

monthly_perf = trend_df.groupby("Order Month").agg(
    Sales=("Sales", "sum"), Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
).sort_index()
print("First months:")
display(monthly_perf.head())
print("Last months:")
display(monthly_perf.tail())
monthly_perf[["Sales", "Profit"]].plot(figsize=(12, 5), marker="o")
plt.title("Monthly Sales and Profit")
plt.xlabel("Order Month")
plt.ylabel("Amount")
plt.tight_layout()
plt.show()

yearly_perf = trend_df.groupby("Order Year").agg(
    Sales=("Sales", "sum"), Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
).sort_index()
display(yearly_perf)
yearly_perf[["Sales", "Profit"]].plot(kind="bar", figsize=(9, 4))
plt.title("Yearly Sales and Profit")
plt.ylabel("Amount")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 12. Discount and profitability

In [ ]:
if {"Discount", "Profit", "Sales"}.issubset(df.columns):
    discount_summary = df.groupby("Discount", dropna=False).agg(
        Records=("Profit", "size"), Sales=("Sales", "sum"),
        Profit=("Profit", "sum"), Average_Profit=("Profit", "mean")
    ).sort_index()
    display(discount_summary)
    plt.figure(figsize=(9, 5))
    sns.scatterplot(data=df, x="Discount", y="Profit", alpha=0.35)
    plt.title("Discount vs Profit")
    plt.tight_layout()
    plt.show()
    print("Observed association does not by itself establish causation.")

## 13. Findings to document after reviewing outputs

Write only findings supported by the tables and charts:
- highest-sales and highest-profit category/sub-category;
- strongest and weakest market or region;
- visible monthly/yearly trends;
- discount levels associated with lower or higher observed profit;
- data-quality limitations, including missing postal codes and mixed date strings.

Avoid claiming causation from descriptive analysis alone.

## 14. Optional export of summary tables

In [ ]:
EXPORT_SUMMARIES = False  # Change to True if you want CSV summary files.

if EXPORT_SUMMARIES:
    reports_dir = PROJECT_ROOT / "reports"
    reports_dir.mkdir(parents=True, exist_ok=True)
    kpi_summary.to_csv(reports_dir / "eda_baseline_kpis.csv", index=False)
    if "category_perf" in globals():
        category_perf.to_csv(reports_dir / "eda_category_performance.csv")
    if "monthly_perf" in globals():
        monthly_perf.to_csv(reports_dir / "eda_monthly_performance.csv")
    print("Summary tables exported to:", reports_dir)
else:
    print("Export disabled; no files written.")